# CS604 项目：ViT-B/16 迁移到 1 万类细粒度分类（Colab 版）

这个 notebook 按项目要求跑完整流程，最后生成可以直接上传 CodaBench 的 `val_team_XX.txt`（已打包成 zip）。

| 步骤 | 内容 |
|---|---|
| 1 | 准备数据：Drive 上有数据就从 Drive 解压，否则从 HF 下载约 44GB（10–20 分钟，不计入训练时间） |
| 2 | 从训练集每类留出少量图片作本地验证集（官方验证集没有标签） |
| 3 | 初始化分类头：类原型（即零样本基线）→ 在缓存特征上做线性探测 |
| 4 | 微调：`full` 全量、`lora`、`adaptformer`、`adapter`、`bitfit`、`partial`（只训练最后几层）、`linear`（只用分类头） |
| 5 | 本地评估、loss 曲线、可训练参数量、推理延迟、训练 GPU 时长、峰值显存 |
| 6 | 预测验证集（Phase 2 可加测试集），生成提交文件 |

**使用方法**
1. `代码执行程序 → 更改运行时类型` 选 **A100 GPU**（推荐）或 L4。T4 也能跑，但很慢。
2. 在下面的配置单元里改 `TEAM_ID` 和 `METHOD`。
3. `代码执行程序 → 全部运行`。会弹出 Google Drive 授权，结果和断点都存在 Drive 里。
4. 跑完后浏览器会自动下载 `submission_val_*.zip`，同一份文件也在 Drive 的 `cs604/runs/<RUN_NAME>/` 里，直接上传 CodaBench 即可。
5. 在同一个会话里做下一个实验：改完配置后再「全部运行」。数据不会重新准备，模型会从预训练权重重新开始，结果存进新的 run 文件夹。

**建议先做一次快速提交**：设 `METHOD = "linear"`、`HEAD_INIT = "proto"`，不需要训练，几分钟就能得到零样本基线（文档报告为 43.7%），可以先确认提交格式没问题。

**几点说明**
- **训练时间预算**：文档规定 2 小时 × 48GB 显存，这里按显存 × 小时 = 96 GB·h 折算。A100-40GB 约 2.4 小时，A100-80GB / H100 约 1.2 小时，L4 约 4 小时。训练开始后会先测速，如果按设定的 epoch 数会超时，就自动压缩学习率日程，保证在预算内训完。
- **参数量口径**：和文档一致，**不含分类头**。LoRA r=8 加在 qkv 上是 294,912 个参数，占 backbone 的 0.344%，和文档里的 0.34% 一致。1 万类分类头另有 7.69M 参数，会单独列出。
- **基础模型**：文档对基础模型的写法前后不一致（IN21K→IN1K 微调版 vs. 只在 IN21K 上预训练的版本），`MODEL_NAME` 里列了两个选项，建议向助教确认用哪个。

## 1. 配置

In [ ]:
# ===== 必改 =====
TEAM_ID = "xx"          # 队号，提交文件名为 val_team_{TEAM_ID}.txt
METHOD = "lora"         # 'lora' | 'full' | 'adaptformer' | 'adapter' | 'bitfit' | 'partial' | 'linear'

# ===== 常用 =====
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"  # 文档第 1 页给的 timm 版本（IN21K→IN1K）
# MODEL_NAME = "vit_base_patch16_224.orig_in21k"            # = HF google/vit-base-patch16-224-in21k（只在 IN21K 上预训练）
HOLDOUT_PER_CLASS = 2   # 每类从训练集留出几张做本地验证；最终冲榜时可以设为 0，用满全部训练数据
EPOCHS = 10             # 最多训练几个 epoch，超出时间预算时会自动缩短
TIME_BUDGET_H = None    # None = 按 96 GB·h 自动折算；也可以直接填小时数
HEAD_INIT = "lp"        # 'lp' 线性探测 | 'proto' 类原型（零样本） | 'random'
TRAIN_HEAD = True       # PEFT 方法是否同时训练分类头
RUN_NAME = None         # None = 自动命名；断线后续训时填上次的 RUN_NAME
LOAD_RUN = None         # 填训练好的 RUN_NAME：跳过训练，只加载权重做预测（Phase 2 用）
PREDICT_SPLITS = ["val"]  # Phase 2 发布测试集后改成 ["test"]（或 ["val", "test"]）

# ===== 超参数（None = 使用下面 METHOD_DEFAULTS 里的值）=====
BATCH_SIZE = None       # 单步 batch；None = 按显存自动选择，再用梯度累积凑到 256
LR = None
WEIGHT_DECAY = None
HEAD_LR = None          # None = 与 LR 相同
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS = 1
RRC_SCALE = (0.25, 1.0)  # RandomResizedCrop 的面积范围
LORA_R = 8
LORA_ALPHA = 16
LORA_TARGETS = ["qkv"]   # 可选 'qkv', 'proj', 'fc1', 'fc2'
ADAPTER_DIM = 64         # AdaptFormer / Adapter 的瓶颈维度
PARTIAL_LAST_K = 2       # 'partial'：只训练最后 K 个 block
LP_EPOCHS = 10           # 线性探测的轮数（在缓存特征上训练，很快）
TTA_FLIP = False         # 预测时加水平翻转（推理延迟翻倍）
COMPILE = False          # torch.compile，A100 上一般能再快 10–20%，首次编译需要 1–2 分钟
CKPT_EVERY = 1           # 每几个 epoch 存一次断点
AUTO_DOWNLOAD = True     # 跑完让浏览器自动下载提交文件；关掉页面在后台运行时改成 False
DISCONNECT_WHEN_DONE = False  # 跑完后自动断开运行时、停止消耗计算单元（后台运行时建议 True）
LOG_EVERY = 50           # 每多少步打印一次日志
SEED = 0

METHOD_DEFAULTS = {
    #                   学习率     权重衰减  逐层学习率衰减      DropPath
    "full":        dict(lr=1e-4, wd=0.05, layer_decay=0.75, drop_path=0.1),
    "partial":     dict(lr=3e-4, wd=0.05, layer_decay=1.0, drop_path=0.0),
    "lora":        dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0),
    "adaptformer": dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0, adapter_scale=0.1),
    "adapter":     dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0, adapter_scale=1.0),
    "bitfit":      dict(lr=1e-3, wd=0.0, layer_decay=1.0, drop_path=0.0),
    "linear":      dict(lr=0.0, wd=0.0, layer_decay=1.0, drop_path=0.0),
}

# ===== 路径 =====
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/cs604"
DATA_DIR = "/content/data"
TAR_SOURCE_DIR = f"{DRIVE_DIR}/data"  # 这里有 tar 包就直接解压（见 cs604_download_to_drive.ipynb），没有就从 HF 下载
HF_REPO = "doem1997/cs604-course-data"
NUM_CLASSES = 10000

## 2. 环境准备

In [ ]:
# Colab 自带 torch / torchvision，这里补装 timm、peft 和更快的 HF 下载后端
!pip install -q timm peft hf_xet

In [ ]:
# 只记录配置单元里的变量；同一会话里再次运行时，不会混进上一次运行留下的状态
CONFIG_KEYS = ("TEAM_ID", "METHOD", "MODEL_NAME", "HOLDOUT_PER_CLASS", "EPOCHS", "TIME_BUDGET_H", "HEAD_INIT",
               "TRAIN_HEAD", "RUN_NAME", "LOAD_RUN", "PREDICT_SPLITS", "BATCH_SIZE", "LR", "WEIGHT_DECAY", "HEAD_LR",
               "LABEL_SMOOTHING", "WARMUP_EPOCHS", "RRC_SCALE", "LORA_R", "LORA_ALPHA", "LORA_TARGETS", "ADAPTER_DIM",
               "PARTIAL_LAST_K", "LP_EPOCHS", "TTA_FLIP", "COMPILE", "CKPT_EVERY", "AUTO_DOWNLOAD",
               "DISCONNECT_WHEN_DONE", "LOG_EVERY", "SEED",
               "METHOD_DEFAULTS", "USE_DRIVE", "DRIVE_DIR", "DATA_DIR", "TAR_SOURCE_DIR", "HF_REPO", "NUM_CLASSES")
CONFIG = {k: globals()[k] for k in CONFIG_KEYS}

import os, re, gc, sys, json, math, time, types, random, shutil, zipfile, subprocess, contextlib, urllib.request
from datetime import datetime

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")  # 加速 HF 大文件下载

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules

# ---- 输出目录：Colab 上放 Google Drive，断线也不会丢 ----
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = DRIVE_DIR
else:
    OUT_ROOT = "/content/outputs" if IN_COLAB else os.path.abspath("outputs")

# ---- 只做预测：沿用训练时的模型结构配置 ----
if LOAD_RUN:
    with open(os.path.join(OUT_ROOT, "runs", LOAD_RUN, "config.json")) as f:
        saved = json.load(f)
    for k in ("METHOD", "MODEL_NAME", "NUM_CLASSES", "HEAD_INIT", "TRAIN_HEAD", "LORA_R", "LORA_ALPHA",
              "LORA_TARGETS", "ADAPTER_DIM", "PARTIAL_LAST_K", "METHOD_DEFAULTS"):
        globals()[k] = CONFIG[k] = saved[k]
    RUN_NAME = LOAD_RUN

# ---- 各方法的默认超参数 ----
MD = METHOD_DEFAULTS[METHOD]
LR = MD["lr"] if LR is None else LR
WEIGHT_DECAY = MD["wd"] if WEIGHT_DECAY is None else WEIGHT_DECAY
HEAD_LR = LR if HEAD_LR is None else HEAD_LR
LAYER_DECAY, DROP_PATH = MD["layer_decay"], MD["drop_path"]
ADAPTER_SCALE = MD.get("adapter_scale", 1.0)

# ---- 设备与混合精度 ----
if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    gib = props.total_memory / 1024**3
    VRAM_GB = next((v for v in (8, 12, 16, 24, 32, 40, 48, 80, 94, 96, 141, 192) if v >= 0.98 * gib), round(gib))  # 标称显存
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4/V100 没有 bf16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
    GPU_NAME, VRAM_GB, AMP_DTYPE = DEVICE, None, None

if BATCH_SIZE is None:
    BATCH_SIZE = 256 if (VRAM_GB or 0) >= 40 else 128 if (VRAM_GB or 0) >= 24 else 64
ACCUM_STEPS = max(1, 256 // BATCH_SIZE)  # 梯度累积，等效 batch 为 256
NUM_WORKERS = min(16, os.cpu_count() or 2)
if TIME_BUDGET_H is None:
    TIME_BUDGET_H = 96 / VRAM_GB if VRAM_GB else 1.0  # 2 h × 48 GB = 96 GB·h

RUN_NAME = RUN_NAME or f"{METHOD}_{datetime.now():%m%d_%H%M}"
OUT_DIR = os.path.join(OUT_ROOT, "runs", RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
CONFIG.update(LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, HEAD_LR=HEAD_LR, BATCH_SIZE=BATCH_SIZE,
              ACCUM_STEPS=ACCUM_STEPS, TIME_BUDGET_H=TIME_BUDGET_H, RUN_NAME=RUN_NAME)
if not LOAD_RUN:
    with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
        json.dump(CONFIG, f, indent=2)


def log(*args):
    """打印，同时追加到 train_log.txt（报告要附训练日志）"""
    msg = " ".join(str(a) for a in args)
    print(msg, flush=True)
    with open(os.path.join(OUT_DIR, "train_log.txt"), "a") as f:
        f.write(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n")


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
log(f"Run {RUN_NAME} -> {OUT_DIR}")
log(f"Device {GPU_NAME} ({VRAM_GB} GB), AMP {AMP_DTYPE}, {os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM, "
    f"{shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free disk")
log(f"Method {METHOD}: lr {LR}, head lr {HEAD_LR}, wd {WEIGHT_DECAY}, layer decay {LAYER_DECAY}, "
    f"batch {BATCH_SIZE}x{ACCUM_STEPS}, time budget {TIME_BUDGET_H:.2f} h")
if AMP_DTYPE == torch.float16:
    log("WARNING: this GPU has no bf16 (e.g. T4); using fp16 + GradScaler, training will be slow.")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. 下载并解压数据

- 如果 `TAR_SOURCE_DIR`（默认是 Drive 的 `cs604/data/`）里有 tar 包，就直接从那里解压，不访问 HF。可以先用 `cs604_download_to_drive.ipynb` 把数据一次性存进 Drive。
- 否则用 `hf_hub_download` 下载 tar 包（支持断点续传），解压后删除 tar 包，释放磁盘。如果磁盘放不下「tar 包 + 解压结果」，就改为边下载边解压。
- 图片保持原图（最长边 500px），不预先缩小：A100 运行时的 CPU 解码速度足够跟上 GPU。
- 解压完成后会写一个标记文件，同一个 Colab 会话里重复运行会直接跳过。换了新会话需要重新准备数据：数据在 Drive 上时只需解压，否则要重新下载。

In [ ]:
from huggingface_hub import hf_hub_download, hf_hub_url


def fetch(fname):
    """返回文件的本地路径：优先用 DATA_DIR / TAR_SOURCE_DIR 里已有的，否则从 HF 下载"""
    for d in (DATA_DIR, TAR_SOURCE_DIR):
        if d and os.path.exists(os.path.join(d, fname)):
            return os.path.join(d, fname)
    try:
        return hf_hub_download(HF_REPO, fname, repo_type="dataset", local_dir=DATA_DIR)
    except Exception as e:  # 个别 huggingface_hub 版本下载小文件时会报解码错误，改用 urllib
        log(f"hf_hub_download({fname}) failed: {e!r}; retrying with urllib")
        dst = os.path.join(DATA_DIR, fname)
        urllib.request.urlretrieve(hf_hub_url(HF_REPO, fname, repo_type="dataset"), dst + ".part")
        os.replace(dst + ".part", dst)
        return dst


def remote_size_gb(fname):
    try:
        from huggingface_hub import HfApi
        return HfApi().get_paths_info(HF_REPO, [fname], repo_type="dataset")[0].size / 1e9
    except Exception:
        return 45.0


def prepare_split(split):
    """把 {split}.tar.gz 解压到 DATA_DIR/{split}/，并核对图片数量"""
    list_path = fetch(f"{split}.txt")
    with open(list_path) as f:
        n_expected = sum(1 for line in f if line.strip())
    marker = os.path.join(DATA_DIR, f".{split}_ready")
    if os.path.exists(marker):
        return list_path
    t0, tar_name = time.time(), f"{split}.tar.gz"
    src = next((os.path.join(d, tar_name) for d in (DATA_DIR, TAR_SOURCE_DIR)
                if d and os.path.exists(os.path.join(d, tar_name))), None)
    if src is None and shutil.disk_usage(DATA_DIR).free / 1e9 < 2.1 * remote_size_gb(tar_name) + 5:
        # 磁盘放不下「tar 包 + 解压结果」：边下载边解压（中途断了需要重来）
        url = hf_hub_url(HF_REPO, tar_name, repo_type="dataset")
        log(f"[{split}] low disk space: streaming download + extract ...")
        subprocess.run(f"set -o pipefail; curl -fsSL --retry 5 '{url}' | tar -xz -C '{DATA_DIR}'",
                       shell=True, executable="/bin/bash", check=True)
    else:
        if src is None:
            log(f"[{split}] downloading {tar_name} ...")
            src = fetch(tar_name)
        log(f"[{split}] extracting {src} ...")
        subprocess.run(["tar", "-xzf", src, "-C", DATA_DIR], check=True)
        if os.path.dirname(os.path.abspath(src)) == os.path.abspath(DATA_DIR):
            os.remove(src)  # 解压完删掉 tar 包
    n_found = sum(len(files) for _, _, files in os.walk(os.path.join(DATA_DIR, split)))
    assert n_found == n_expected, f"{split}: found {n_found} images, expected {n_expected}"
    open(marker, "w").close()
    log(f"[{split}] {n_found} images ready in {(time.time() - t0) / 60:.1f} min")
    return list_path


TRAIN_LIST = None if LOAD_RUN else prepare_split("train")  # 只做预测时不需要训练集
PRED_LISTS = {split: prepare_split(split) for split in PREDICT_SPLITS}

## 4. 模型与微调方法

- `lora` 用 Hugging Face [PEFT](https://github.com/huggingface/peft) 实现，训练完会把 LoRA 权重合并回原矩阵，推理时没有额外开销。
- `adaptformer` 按 [AdaptFormer 官方实现](https://github.com/ShoufaChen/AdaptFormer)：瓶颈 64，缩放 0.1，与 FFN 并联，上投影零初始化。
- `adapter` 是串行 Adapter，接在 FFN 之后。
- `bitfit` 只训练 bias，`partial` 只训练最后 `PARTIAL_LAST_K` 个 block，`linear` 只用分类头。

ViT-B/16 上各方法的可训练参数（默认配置，不含分类头；backbone 共 85,798,656 个参数）：

| 方法 | 可训练参数 | 占 backbone | 推理时额外参数 |
|---|---|---|---|
| `full` | 85,798,656 | 100% | 0 |
| `lora`（r=8，qkv） | 294,912 | 0.344% | 0（已合并） |
| `adaptformer` / `adapter`（瓶颈 64） | 1,189,632 | 1.387% | 1,189,632 |
| `bitfit` | 102,912 | 0.120% | 0 |
| `partial`（最后 2 个 block） | 14,177,280 | 16.5%（超过文档建议的 5%） | 0 |
| `linear` | 0 | 0% | 0 |

1 万类分类头另有 7,690,000 个参数，报告里需要单独说明它是否参与训练。

In [ ]:
class Adapter(nn.Module):
    """瓶颈 MLP：down → ReLU → up。up 零初始化，所以插入后模型输出一开始不变（同 AdaptFormer 官方实现）"""

    def __init__(self, dim, bottleneck, scale, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.up = nn.Linear(bottleneck, dim)
        self.drop = nn.Dropout(dropout)
        self.scale = scale
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.drop(F.relu(self.down(x)))) * self.scale


def adaptformer_block_forward(self, x, *args, **kwargs):
    # AdaptFormer：adapter 与 FFN 并联，输入是进入 FFN 分支之前的残差流
    x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), *args, **kwargs)))
    return x + self.drop_path2(self.ls2(self.mlp(self.norm2(x)))) + self.adapter(x)


def adapter_block_forward(self, x, *args, **kwargs):
    # 串行 Adapter：接在 FFN 输出之后，自带残差
    x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), *args, **kwargs)))
    h = self.mlp(self.norm2(x))
    return x + self.drop_path2(self.ls2(h + self.adapter(h)))


LORA_MODULES = {"qkv": r"attn\.qkv", "proj": r"attn\.proj", "fc1": r"mlp\.fc1", "fc2": r"mlp\.fc2"}


def is_head(name):
    return re.search(r"(^|\.)head\.(weight|bias)$", name) is not None


def unwrap(net):
    return net.get_base_model() if hasattr(net, "get_base_model") else net


def apply_method(net):
    """按 METHOD 冻结参数、插入 PEFT 模块。返回的模型可能被 peft 包了一层"""
    if METHOD == "full":
        return net.requires_grad_(True)
    net.requires_grad_(False)
    if METHOD == "lora":
        from peft import LoraConfig, get_peft_model
        pattern = r"blocks\.\d+\.(" + "|".join(LORA_MODULES[t] for t in LORA_TARGETS) + ")"
        net = get_peft_model(net, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                                             target_modules=pattern, bias="none"))
    elif METHOD in ("adaptformer", "adapter"):
        forward = adaptformer_block_forward if METHOD == "adaptformer" else adapter_block_forward
        for blk in net.blocks:
            blk.adapter = Adapter(net.embed_dim, ADAPTER_DIM, ADAPTER_SCALE)
            blk.forward = types.MethodType(forward, blk)
    elif METHOD == "bitfit":
        for n, p in net.named_parameters():
            if n.endswith(".bias"):
                p.requires_grad_(True)
    elif METHOD == "partial":
        for blk in net.blocks[-PARTIAL_LAST_K:]:
            blk.requires_grad_(True)
        net.norm.requires_grad_(True)
    unwrap(net).head.requires_grad_(TRAIN_HEAD and METHOD != "linear")
    return net


def param_report(net):
    named = list(net.named_parameters())
    head_trained = any(p.requires_grad for n, p in named if is_head(n))
    n_train = sum(p.numel() for n, p in named if p.requires_grad and not is_head(n))
    source = HEAD_STATS.get("head_source", HEAD_INIT)
    head_mode = ("fine-tuned" if head_trained else "linear probe" if source == "lp"
                 else "class prototypes (not trained)" if source == "proto" else "random, frozen")
    return {"backbone_params": N_BACKBONE, "head_params": N_HEAD, "head_mode": head_mode,
            "trainable_excl_head": n_train, "trainable_pct": 100 * n_train / N_BACKBONE,
            "trainable_incl_head": n_train + (N_HEAD if head_trained or source == "lp" else 0)}


# 同一会话里再次运行时：先释放上一次的模型、优化器和数据加载进程，并重置显存峰值统计
for _name in ("model", "train_model", "optimizer", "scaler", "params", "train_loader", "holdout_loader"):
    globals().pop(_name, None)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_path_rate=DROP_PATH).to(DEVICE)
DATA_CFG = timm.data.resolve_model_data_config(model)
N_BACKBONE = sum(p.numel() for n, p in model.named_parameters() if not is_head(n))
N_HEAD = sum(p.numel() for p in model.head.parameters())
MEAN = torch.tensor(DATA_CFG["mean"], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor(DATA_CFG["std"], device=DEVICE).view(1, 3, 1, 1) * 255
log(f"Model {MODEL_NAME}: backbone {N_BACKBONE:,} params, {NUM_CLASSES}-class head {N_HEAD:,} params")
log(f"Data config: {DATA_CFG}")

## 5. 数据集与本地验证集

官方验证集没有标签，所以从训练集每类留出 `HOLDOUT_PER_CLASS` 张图作本地验证集，用来调参和做消融实验。划分由 `SEED` 固定，每次运行都一样。

In [ ]:
def read_list(path):
    items = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if parts:
                items.append((parts[0], int(parts[1]) if len(parts) > 1 else -1))
    return items


def split_holdout(items, k, seed):
    by_class = {}
    for path, label in items:
        by_class.setdefault(label, []).append(path)
    rng = random.Random(seed)
    train, holdout = [], []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        holdout += [(p, label) for p in paths[:k]]
        train += [(p, label) for p in paths[k:]]
    return train, holdout


class ImageList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(os.path.join(DATA_DIR, path)) as im:
            img = im.convert("RGB")
        return self.transform(img), label


IMG_SIZE = DATA_CFG["input_size"][-1]
BICUBIC = T.InterpolationMode.BICUBIC
# 在 CPU 上只做裁剪/翻转，输出 uint8；归一化放到 GPU 上做，减轻 CPU 负担
TRAIN_TF = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=RRC_SCALE, interpolation=BICUBIC),
                      T.RandomHorizontalFlip(), T.PILToTensor()])
EVAL_TF = T.Compose([T.Resize(int(IMG_SIZE / DATA_CFG["crop_pct"]), interpolation=BICUBIC),
                     T.CenterCrop(IMG_SIZE), T.PILToTensor()])


def make_loader(items, train, shuffle=False):
    return DataLoader(ImageList(items, TRAIN_TF if train else EVAL_TF),
                      batch_size=BATCH_SIZE if train else 2 * BATCH_SIZE, shuffle=shuffle, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda",
                      persistent_workers=train and NUM_WORKERS > 0)


def to_input(x):
    return (x.to(DEVICE, non_blocking=True).float() - MEAN) / STD


def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if DEVICE == "cuda" else contextlib.nullcontext()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


if LOAD_RUN:
    train_items, holdout_items = [], []
else:
    train_items, holdout_items = split_holdout(read_list(TRAIN_LIST), HOLDOUT_PER_CLASS, SEED)
pred_items = {split: read_list(path) for split, path in PRED_LISTS.items()}
log(f"train {len(train_items):,} | holdout {len(holdout_items):,} | "
    + " | ".join(f"{s} {len(v):,}" for s, v in pred_items.items()) + f" | {NUM_WORKERS} loader workers")

## 6. 分类头初始化

对应文档提示里的 *Initialization*。新的 1 万类分类头如果随机初始化，训练初期的梯度会破坏预训练特征。这里先用冻结的 backbone 提取一遍训练集特征：

1. **类原型**：每类特征均值（L2 归一化）作为分类头权重，这就是文档里的 "Zero-Shot with initialization of classifier head"（0 个可训练参数）。
2. **线性探测**：以原型为起点，在缓存的特征上训练分类头，几秒钟就能完成。如果在本地验证集上不如原型，就保留原型。

这一步的耗时计入训练 GPU 时长。续训或只做预测时会跳过。

In [ ]:
LP_LR, LP_WD, LP_BATCH = 1e-3, 1e-4, 4096


class Clock:
    """累计 GPU 工作时间（分类头初始化 + 训练 + 训练中的验证），不含数据准备和存断点"""

    def __init__(self):
        self.used, self._t0 = 0.0, None

    def __enter__(self):
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        sync()
        self.used += time.time() - self._t0
        self._t0 = None

    @property
    def elapsed(self):
        return self.used + (time.time() - self._t0 if self._t0 else 0.0)


@torch.inference_mode()
def extract_features(net, items, desc):
    net.eval()
    feats, labels = [], []
    for x, y in tqdm(make_loader(items, train=False), desc=desc, leave=False):
        with autocast():
            f = net.forward_head(net.forward_features(to_input(x)), pre_logits=True)
        feats.append(f.half())
        labels.append(y.to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


@torch.inference_mode()
def class_prototypes(feats, labels):
    protos = torch.zeros(NUM_CLASSES, feats.shape[1], device=feats.device)
    for i in range(0, len(feats), 65536):
        protos.index_add_(0, labels[i:i + 65536], F.normalize(feats[i:i + 65536].float(), dim=1))
    return F.normalize(protos, dim=1)


@torch.inference_mode()
def fit_logit_scale(protos, feats, labels, n=10000):
    """给余弦相似度挑一个合适的温度，让原型头的 softmax 不至于太平或太尖"""
    idx = torch.randperm(len(feats), device=feats.device)[:n]
    cos = F.normalize(feats[idx].float(), dim=1) @ protos.T
    losses = {s: F.cross_entropy(cos * s, labels[idx]).item() for s in (5, 10, 15, 20, 30, 40, 60, 80)}
    return min(losses, key=losses.get)


@torch.inference_mode()
def topk_accuracy(logits_fn, feats, labels, chunk=4096):
    c1 = c5 = 0
    for i in range(0, len(feats), chunk):
        top5 = logits_fn(feats[i:i + chunk].float()).topk(5, dim=1).indices
        y = labels[i:i + chunk]
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
    return c1 / len(feats), c5 / len(feats)


def linear_probe(feats, labels, weight, bias):
    head = nn.Linear(feats.shape[1], NUM_CLASSES).to(DEVICE)
    with torch.no_grad():
        head.weight.copy_(weight)
        head.bias.copy_(bias)
    opt = torch.optim.AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WD)
    total = LP_EPOCHS * math.ceil(len(feats) / LP_BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / total)))
    for _ in tqdm(range(LP_EPOCHS), desc="linear probe", leave=False):
        perm = torch.randperm(len(feats), device=feats.device)
        for i in range(0, len(feats), LP_BATCH):
            idx = perm[i:i + LP_BATCH]
            loss = F.cross_entropy(head(feats[idx].float()), labels[idx], label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
    return head.weight.detach(), head.bias.detach()


clock = Clock()
HEAD_STATS = {}
RESUME_CKPT = os.path.join(OUT_DIR, "last.pt")
RESUME_CKPT = RESUME_CKPT if not LOAD_RUN and os.path.exists(RESUME_CKPT) else None

if LOAD_RUN or RESUME_CKPT or HEAD_INIT == "random":
    reason = "predict-only run" if LOAD_RUN else "resuming from checkpoint" if RESUME_CKPT else "HEAD_INIT='random'"
    log(f"Skipping head initialization ({reason})")
else:
    with clock:
        f_tr, y_tr = extract_features(model, train_items, "train features")
        protos = class_prototypes(f_tr, y_tr)
        scale = fit_logit_scale(protos, f_tr, y_tr)
        # 线性头作用在未归一化的特征上：除以平均特征模长，让 logits ≈ scale × 余弦相似度
        weight = protos * (scale / f_tr.float().norm(dim=1).mean())
        bias = torch.zeros(NUM_CLASSES, device=DEVICE)
        HEAD_STATS.update(head_source="proto", proto_logit_scale=scale)
        if holdout_items:
            f_ho, y_ho = extract_features(model, holdout_items, "holdout features")
            HEAD_STATS["proto_top1"], HEAD_STATS["proto_top5"] = topk_accuracy(lambda f: f @ weight.T, f_ho, y_ho)
            log(f"Prototype head (zero-shot, 0 trainable params): holdout top-1 {HEAD_STATS['proto_top1']:.2%}")
        if HEAD_INIT == "lp":
            lp_w, lp_b = linear_probe(f_tr, y_tr, weight, bias)
            if holdout_items:
                HEAD_STATS["lp_top1"], HEAD_STATS["lp_top5"] = topk_accuracy(lambda f: f @ lp_w.T + lp_b, f_ho, y_ho)
                log(f"Linear-probe head: holdout top-1 {HEAD_STATS['lp_top1']:.2%}")
            if not holdout_items or HEAD_STATS["lp_top1"] >= HEAD_STATS["proto_top1"]:
                weight, bias = lp_w, lp_b
                HEAD_STATS["head_source"] = "lp"
            else:
                log("Linear probe is worse than the prototypes on holdout; keeping the prototype head")
        with torch.no_grad():
            model.head.weight.copy_(weight)
            model.head.bias.copy_(bias)
        del f_tr, y_tr
        if holdout_items:
            del f_ho, y_ho
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    log(f"Head initialized from {HEAD_STATS['head_source']} in {clock.used / 60:.1f} min")

## 7. 训练

- AdamW，前 `WARMUP_EPOCHS` 个 epoch 线性 warmup，之后余弦衰减。bf16 混合精度，梯度裁剪 1.0，label smoothing 0.1。
- 前 120 步测速后，如果按 `EPOCHS` 跑会超出时间预算，就把学习率日程压缩到预算内能跑完的步数。
- 每个 epoch 结束时在本地验证集上评估一次，并把断点存到 Drive。断线后把 `RUN_NAME` 设为同一个名字，重新「全部运行」，就会从最近的断点继续。

In [ ]:
model = apply_method(model).to(DEVICE)
p0 = param_report(model)
log(f"Trainable (excl. head): {p0['trainable_excl_head']:,} = {p0['trainable_pct']:.3f}% of backbone | "
    f"head {N_HEAD:,} params ({p0['head_mode']})")


def build_param_groups(net):
    """分类头用 HEAD_LR；其余参数按层衰减学习率（只有 full 默认开启）；bias、norm、位置编码不做 weight decay"""
    num_layers = len(unwrap(net).blocks) + 1
    groups = {}
    for n, p in net.named_parameters():
        if not p.requires_grad:
            continue
        name = n.replace("base_model.model.", "")
        if is_head(name):
            lr = HEAD_LR
        else:
            m = re.match(r"blocks\.(\d+)\.", name)
            if m:
                layer = int(m.group(1)) + 1
            elif name.startswith(("cls_token", "pos_embed", "reg_token", "patch_embed")):
                layer = 0
            else:
                layer = num_layers
            lr = LR * LAYER_DECAY ** (num_layers - layer)
        wd = 0.0 if p.ndim <= 1 or name in ("cls_token", "pos_embed", "reg_token") else WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd, "base_lr": lr} for (lr, wd), ps in groups.items()]


def lr_factor(s):
    if s < WARMUP_STEPS:
        return (s + 1) / WARMUP_STEPS
    progress = (s - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))


@torch.inference_mode()
def evaluate(net, loader):
    net.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        with autocast():
            top5 = net(to_input(x)).topk(5, dim=1).indices
        y = y.to(DEVICE)
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return c1 / n, c5 / n


def save_checkpoint(path, **extra):
    """只存可训练参数和分类头（LoRA 约 30MB，full 约 370MB）"""
    state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad or is_head(n)}
    torch.save({"state": state, **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


steps_per_epoch = len(train_items) // (BATCH_SIZE * ACCUM_STEPS)
TOTAL_STEPS = 0 if (LOAD_RUN or METHOD == "linear") else EPOCHS * steps_per_epoch
WARMUP_STEPS = max(1, int(WARMUP_EPOCHS * steps_per_epoch))
BUDGET_S = TIME_BUDGET_H * 3600
step, start_epoch = 0, 0
hist = {"step": [], "loss": [], "lr": [], "epoch": [], "epoch_loss": [], "holdout_top1": []}

if LOAD_RUN:
    ck = torch.load(os.path.join(OUT_DIR, "final.pt"), map_location="cpu", weights_only=False)
    _, unexpected = model.load_state_dict(ck["state"], strict=False)
    assert not unexpected, f"unexpected keys: {unexpected[:5]}"
    log(f"Loaded trained weights from {OUT_DIR}/final.pt")
elif TOTAL_STEPS:
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(build_param_groups(model), **({"fused": True} if DEVICE == "cuda" else {}))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    if RESUME_CKPT:
        ck = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
        _, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not unexpected, f"unexpected keys: {unexpected[:5]}"
        optimizer.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
        step, start_epoch = ck["step"], ck["epoch"] + 1
        TOTAL_STEPS, WARMUP_STEPS, clock.used = ck["total_steps"], ck["warmup_steps"], ck["clock"]
        hist, HEAD_STATS = ck["hist"], ck["head_stats"]
        log(f"Resumed at epoch {start_epoch + 1}, step {step}, {clock.used / 3600:.2f} GPU-hours already used")

    train_model = torch.compile(model) if COMPILE and DEVICE == "cuda" else model
    train_loader = make_loader(train_items, train=True, shuffle=True)
    holdout_loader = make_loader(holdout_items, train=False) if holdout_items else None
    log(f"Training up to {EPOCHS} epochs x {steps_per_epoch} steps (warmup {WARMUP_STEPS} steps)")
    first_step, t_mark, t_start, done = step, 0.0, clock.elapsed, step >= TOTAL_STEPS
    for epoch in range(start_epoch, EPOCHS):
        if done:
            break
        train_model.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, run_n, ep_loss, ep_n, t_log = 0.0, 0, 0.0, 0, time.time()
        with clock:
            for i, (x, y) in enumerate(train_loader):
                with autocast():
                    loss = F.cross_entropy(train_model(to_input(x)), y.to(DEVICE, non_blocking=True),
                                           label_smoothing=LABEL_SMOOTHING)
                scaler.scale(loss / ACCUM_STEPS).backward()
                run_loss += loss.detach()
                run_n += 1
                if (i + 1) % ACCUM_STEPS:
                    continue
                for g in optimizer.param_groups:
                    g["lr"] = g["base_lr"] * lr_factor(step)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

                # 测速：按实际速度把学习率日程压缩到时间预算以内
                if step - first_step == 20:
                    t_mark = clock.elapsed
                elif step - first_step == 120:
                    sec_per_step = (clock.elapsed - t_mark) / 100
                    fit = step + int((0.97 * BUDGET_S - clock.elapsed - 30 * (EPOCHS - epoch)) / sec_per_step)
                    log(f"Speed {BATCH_SIZE * ACCUM_STEPS / sec_per_step:.0f} img/s, "
                        f"{sec_per_step * steps_per_epoch / 60:.1f} min/epoch")
                    if fit < TOTAL_STEPS:
                        TOTAL_STEPS = max(fit, step + 1)
                        WARMUP_STEPS = min(WARMUP_STEPS, max(1, TOTAL_STEPS // 10))
                        log(f"Time budget fits ~{TOTAL_STEPS / steps_per_epoch:.1f} epochs: "
                            f"LR schedule compressed to {TOTAL_STEPS} steps")

                if step % LOG_EVERY == 0:
                    avg = (run_loss / run_n).item()
                    if not math.isfinite(avg):
                        raise RuntimeError("Loss is NaN/inf; try a lower LR")
                    eta = (TOTAL_STEPS - step) * (clock.elapsed - t_start) / (step - first_step)
                    log(f"ep {epoch + 1}/{EPOCHS} step {step}/{TOTAL_STEPS} | loss {avg:.4f} | "
                        f"lr {LR * lr_factor(step):.2e} | {run_n * BATCH_SIZE / (time.time() - t_log):.0f} img/s | "
                        f"{clock.elapsed / 60:.1f} min | ETA {eta / 60:.0f} min")
                    hist["step"].append(step)
                    hist["loss"].append(avg)
                    hist["lr"].append(LR * lr_factor(step))
                    ep_loss += avg * run_n
                    ep_n += run_n
                    run_loss, run_n, t_log = 0.0, 0, time.time()
                if step >= TOTAL_STEPS or clock.elapsed > 1.05 * BUDGET_S:
                    done = True
                    break
            if run_n:
                ep_loss += float(run_loss)
                ep_n += run_n
            top1 = evaluate(train_model, holdout_loader)[0] if holdout_loader else None
        hist["epoch"].append(epoch + 1)
        hist["epoch_loss"].append(ep_loss / max(1, ep_n))
        hist["holdout_top1"].append(top1)
        log(f"== epoch {epoch + 1}: train loss {ep_loss / max(1, ep_n):.4f}"
            + (f", holdout top-1 {top1:.2%}" if top1 is not None else "") + f", {clock.elapsed / 60:.1f} min used")
        if (epoch + 1) % CKPT_EVERY == 0 or done or epoch + 1 == EPOCHS:
            save_checkpoint(os.path.join(OUT_DIR, "last.pt"), optimizer=optimizer.state_dict(),
                            scaler=scaler.state_dict(), step=step, epoch=epoch, total_steps=TOTAL_STEPS,
                            warmup_steps=WARMUP_STEPS, clock=clock.used, hist=hist, head_stats=HEAD_STATS)
    del train_loader, holdout_loader

if not LOAD_RUN:
    save_checkpoint(os.path.join(OUT_DIR, "final.pt"), config=CONFIG, head_stats=HEAD_STATS)
    with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
        json.dump(hist, f)
    log(f"Training done: {step} steps, {clock.elapsed / 3600:.2f} GPU-hours (budget {TIME_BUDGET_H:.2f} h)")

PARAMS = param_report(model)  # 续训时 HEAD_STATS 从断点恢复，这里重新统计
if METHOD == "lora":
    model = model.merge_and_unload()  # LoRA 权重并回原矩阵，推理时没有额外开销
PARAMS["extra_params_at_inference"] = sum(p.numel() for p in model.parameters()) - N_BACKBONE - N_HEAD

## 8. 评估与训练曲线

In [ ]:
FINAL = {}
if holdout_items:
    FINAL["holdout_top1"], FINAL["holdout_top5"] = evaluate(model, make_loader(holdout_items, train=False))
    log(f"Final holdout: top-1 {FINAL['holdout_top1']:.2%}, top-5 {FINAL['holdout_top5']:.2%}")

INK, MUTED, GRID, SURFACE, SERIES = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb", "#2a78d6"


def style_axis(ax, title, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)


if hist["step"]:
    accs = [(e, 100 * a) for e, a in zip(hist["epoch"], hist["holdout_top1"]) if a is not None]
    fig, axes = plt.subplots(1, 2 if accs else 1, figsize=(11 if accs else 5.5, 3.6), facecolor=SURFACE, squeeze=False)
    ax = axes[0][0]
    ax.plot(hist["step"], hist["loss"], color=SERIES, linewidth=1.5, solid_capstyle="round")
    style_axis(ax, f"Training loss ({METHOD})", "optimizer step", f"cross-entropy (label smoothing {LABEL_SMOOTHING})")
    if accs:
        ax = axes[0][1]
        ep, acc = zip(*accs)
        ax.plot(ep, acc, color=SERIES, linewidth=1.5, marker="o", markersize=6,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        ax.annotate(f"{acc[-1]:.1f}%", (ep[-1], acc[-1]), textcoords="offset points", xytext=(0, 8),
                    ha="center", color=INK, fontsize=9)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        style_axis(ax, "Holdout top-1 accuracy", "epoch", "accuracy (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "curves.png"), dpi=150, facecolor=SURFACE)
    plt.show()

## 9. 推理延迟

用随机输入测量纯模型前向时间（不含图片解码），与训练时使用相同的混合精度。报告里请写明 GPU 型号、batch size 和精度。开启 `TTA_FLIP` 时，单张延迟按两次前向计算。

In [ ]:
@torch.inference_mode()
def measure_latency(net, batch_size, iters, warmup=10):
    net.eval()
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    for _ in range(warmup):
        with autocast():
            net(x)
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        with autocast():
            net(x)
    sync()
    return (time.perf_counter() - t0) / iters


big_batch = 256 if DEVICE == "cuda" else 8
lat_1 = measure_latency(model, 1, iters=100 if DEVICE == "cuda" else 5) * (2 if TTA_FLIP else 1)
lat_b = measure_latency(model, big_batch, iters=20 if DEVICE == "cuda" else 2) * (2 if TTA_FLIP else 1)
FINAL.update(latency_ms_bs1=lat_1 * 1e3, throughput_img_s=big_batch / lat_b,
             latency_setting=f"{GPU_NAME}, {str(AMP_DTYPE).replace('torch.', '') if AMP_DTYPE else 'fp32'}, "
                             f"{'with' if TTA_FLIP else 'no'} flip TTA")
log(f"Inference latency (batch 1): {lat_1 * 1e3:.2f} ms/img | throughput (batch {big_batch}): "
    f"{big_batch / lat_b:.0f} img/s | {FINAL['latency_setting']}")

## 10. 生成 CodaBench 提交文件

In [ ]:
@torch.inference_mode()
def predict(net, items):
    net.eval()
    preds = []
    for x, _ in tqdm(make_loader(items, train=False), desc="predict", leave=False):
        x = to_input(x)
        with autocast():
            logits = net(x).float()
            if TTA_FLIP:
                logits += net(torch.flip(x, dims=[3])).float()
        preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(preds).tolist()


def write_submission(split, items, preds):
    assert len(preds) == len(items) == len({p for p, _ in items}), "every image must appear exactly once"
    assert all(0 <= c < NUM_CLASSES for c in preds)
    name = f"{split}_team_{TEAM_ID}.txt"
    txt_path = os.path.join(OUT_DIR, name)
    with open(txt_path, "w") as f:
        f.writelines(f"{path} {c}\n" for (path, _), c in zip(items, preds))
    zip_path = os.path.join(OUT_DIR, f"submission_{split}_{RUN_NAME}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(txt_path, arcname=name)
    return zip_path


SUBMISSIONS = {}
for split, items in pred_items.items():
    preds = predict(model, items)
    SUBMISSIONS[split] = write_submission(split, items, preds)
    counts = np.bincount(preds, minlength=NUM_CLASSES)
    log(f"[{split}] {len(preds):,} predictions over {np.count_nonzero(counts):,} classes "
        f"(max {counts.max()} per class) -> {SUBMISSIONS[split]}")

## 11. 结果汇总（写报告用）

In [ ]:
summary_path = os.path.join(OUT_DIR, "summary.json")
SUMMARY = {}
if os.path.exists(summary_path):
    with open(summary_path) as f:
        SUMMARY = json.load(f)
SUMMARY.update(run=RUN_NAME, method=METHOD, model=MODEL_NAME, **PARAMS, **FINAL)
if not LOAD_RUN:
    SUMMARY.update(HEAD_STATS, gpu=GPU_NAME, vram_gb=VRAM_GB, batch=f"{BATCH_SIZE}x{ACCUM_STEPS}",
                   amp=str(AMP_DTYPE).replace("torch.", "") if AMP_DTYPE else "fp32",
                   epochs_trained=round(step / max(1, steps_per_epoch), 2), steps=step,
                   gpu_hours=clock.elapsed / 3600, budget_hours=TIME_BUDGET_H,
                   peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None)
SUMMARY.setdefault("submissions", {}).update(SUBMISSIONS)
with open(summary_path, "w") as f:
    json.dump(SUMMARY, f, indent=2)


def pct(key):
    return f"{SUMMARY[key]:.2%}" if SUMMARY.get(key) is not None else "–"


rows = [
    ("Holdout top-1 / top-5", f"{pct('holdout_top1')} / {pct('holdout_top5')}"),
    ("Zero-shot prototype head (holdout top-1)", pct("proto_top1")),
    ("Linear-probe head (holdout top-1)", pct("lp_top1")),
    ("Trainable params (excl. head)",
     f"{SUMMARY['trainable_excl_head']:,} ({SUMMARY['trainable_pct']:.3f}% of {SUMMARY['backbone_params']:,})"),
    ("Classifier head", f"{SUMMARY['head_params']:,} ({SUMMARY['head_mode']})"),
    ("Extra params at inference", f"{SUMMARY['extra_params_at_inference']:,}"),
    ("Inference latency (batch 1)", f"{SUMMARY['latency_ms_bs1']:.2f} ms ({SUMMARY['latency_setting']})"),
    ("Throughput", f"{SUMMARY['throughput_img_s']:.0f} img/s"),
]
if SUMMARY.get("gpu_hours") is not None:
    rows += [
        ("Training GPU-hours", f"{SUMMARY['gpu_hours']:.2f} h on 1× {SUMMARY['gpu']} (budget {SUMMARY['budget_hours']:.2f} h)"),
        ("Epochs / steps / batch", f"{SUMMARY['epochs_trained']} / {SUMMARY['steps']} / {SUMMARY['batch']}"),
    ]
if SUMMARY.get("peak_vram_gb"):
    rows.append(("Peak VRAM (allocated)", f"{SUMMARY['peak_vram_gb']:.1f} GB"))
rows += [(f"Submission ({s})", os.path.basename(p)) for s, p in SUMMARY["submissions"].items()]
table = "| Metric | Value |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in rows)
log("\n" + table)
try:
    from IPython.display import Markdown, display
    display(Markdown(table))
except ImportError:
    pass

## 12. 下载提交文件 / 自动断开

结果此时已经全部存进 Drive。浏览器下载放在最后一步：后台运行时没有浏览器响应，这一步可能一直卡住，所以关掉页面运行时请设 `AUTO_DOWNLOAD = False`。`DISCONNECT_WHEN_DONE = True` 会在确认数据写入 Drive 后释放运行时，停止消耗计算单元；同一个会话里还要接着跑实验时不要打开。

In [ ]:
if IN_COLAB and AUTO_DOWNLOAD:
    from google.colab import files
    for path in SUBMISSIONS.values():
        files.download(path)
if IN_COLAB and DISCONNECT_WHEN_DONE:
    from google.colab import drive, runtime
    log("Run finished; releasing the runtime ...")
    if USE_DRIVE:
        drive.flush_and_unmount()  # 确认所有结果都已上传到 Drive
    runtime.unassign()

## 下一步

- **提交**：把下载的 `submission_val_*.zip` 上传到 CodaBench（每天最多 5 次）。
- **对比实验**：分别用 `METHOD = "full"`、`"lora"`、`"adaptformer"` 等各跑一次。每个 run 的 `summary.json`、`curves.png`、`history.json` 和 `train_log.txt` 都在 Drive 的 `cs604/runs/` 下，可以直接用于报告里的对比表、loss 曲线和训练日志。
- **消融实验思路**：分类头初始化方式（`HEAD_INIT` = random / proto / lp）、LoRA 的 rank 和插入位置（`LORA_R`、`LORA_TARGETS`）、Adapter 维度、label smoothing、训练轮数等。
- **最终冲榜**：确定配置后设 `HOLDOUT_PER_CLASS = 0`，用满每类 45 张重新训练一次。
- **断线续训**：把 `RUN_NAME` 设为断线的那个 run，重新「全部运行」。数据需要重新下载解压，训练会从最近的 epoch 断点继续。
- **Phase 2（10/31 发布测试集）**：设 `LOAD_RUN = "<训练好的 RUN_NAME>"`、`PREDICT_SPLITS = ["test"]`，再「全部运行」即可生成 `test_team_XX.txt`。不需要重新训练，也不会下载训练集。